In [1]:
import pandas as pd
import numpy as np
from google.colab import files
import uuid

In [2]:
# Step 1: Upload and load dataset
uploaded = files.upload()
filename = list(uploaded.keys())[0]  # Get the first uploaded file
df = pd.read_csv(filename)

Saving USA Stock Prediction Prices.csv to USA Stock Prediction Prices.csv


In [3]:
# Dynamically handle dataset shape
num_rows, num_cols = df.shape
if num_rows < 2 or num_cols < 2:
    raise ValueError(f"Dataset must have at least 2 rows and 2 columns, got {df.shape}")

# Ensure timestamp column contains sequential integers starting from 1
if not df.iloc[:, 0].equals(pd.Series(range(1, num_rows + 1))):
    raise ValueError("Timestamp column must contain sequential integers starting from 1")

# Rename first column to 'timestamp' for consistency
df.columns = ['timestamp'] + list(df.columns[1:])

# Step 2: Analyze each price column (all columns except timestamp)
price_columns = df.columns[1:]  # All columns except timestamp
results = []

for col in price_columns:
    prices = df[col]
    max_price = prices.max()
    min_price = prices.min()
    max_idx = prices.idxmax()
    min_idx = prices.idxmin()
    max_time = df['timestamp'].iloc[max_idx]
    min_time = df['timestamp'].iloc[min_idx]
    std_dev = prices.std()

    results.append({
        'asset': col,
        'max_price': max_price,
        'min_price': min_price,
        'max_time': max_time,
        'min_time': min_time,
        'std_dev': std_dev
    })

# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Step 3: Identify buy and sell portfolios
sell_portfolio = results_df[results_df['max_time'] < results_df['min_time']]['asset'].tolist()
buy_portfolio = results_df[results_df['max_time'] > results_df['min_time']]['asset'].tolist()

# Step 4: Calculate weights using inverse of standard deviations
results_df['inv_std'] = 1 / results_df['std_dev'].replace(0, np.inf)
total_inv_std = results_df['inv_std'].sum()
results_df['weight'] = results_df['inv_std'] / total_inv_std

# Step 5: Allocate total investment ($1,000)
total_investment = 1000

# Calculate total weight for buy and sell portfolios
buy_weights_sum = results_df[results_df['asset'].isin(buy_portfolio)]['weight'].sum()
sell_weights_sum = results_df[results_df['asset'].isin(sell_portfolio)]['weight'].sum()

# Normalize weights to ensure full allocation
total_weight = buy_weights_sum + sell_weights_sum
if total_weight > 0:
    buy_allocation = total_investment * (buy_weights_sum / total_weight)
    sell_allocation = total_investment * (sell_weights_sum / total_weight)
else:
    buy_allocation = 0
    sell_allocation = 0

# Step 6: Reallocate weights within each portfolio
buy_weights = results_df[results_df['asset'].isin(buy_portfolio)][['asset', 'weight']].copy()
sell_weights = results_df[results_df['asset'].isin(sell_portfolio)][['asset', 'weight']].copy()

if buy_weights_sum > 0:
    buy_weights['portfolio_weight'] = buy_weights['weight'] / buy_weights_sum
else:
    buy_weights['portfolio_weight'] = 0

if sell_weights_sum > 0:
    sell_weights['portfolio_weight'] = sell_weights['weight'] / sell_weights_sum
else:
    sell_weights['portfolio_weight'] = 0

# Step 7: Calculate integer shares and balance
portfolio = []
balance = total_investment

# Buy portfolio: Buy at min price, sell at max price
for _, row in buy_weights.iterrows():
    if row['portfolio_weight'] > 0:
        asset = row['asset']
        min_price = results_df[results_df['asset'] == asset]['min_price'].iloc[0]
        max_price = results_df[results_df['asset'] == asset]['max_price'].iloc[0]
        allocation = buy_allocation * row['portfolio_weight']
        shares = int(allocation / min_price)  # Integer shares
        if shares > 0:  # Only include if shares are allocated
            cost = shares * min_price
            balance -= cost
            portfolio.append({
                'asset': asset,
                'type': 'buy',
                'shares': shares,
                'buy_price': min_price,
                'sell_price': max_price,
                'buy_time': results_df[results_df['asset'] == asset]['min_time'].iloc[0],
                'sell_time': results_df[results_df['asset'] == asset]['max_time'].iloc[0],
                'cost': cost,
                'revenue': shares * (max_price - min_price)  # Profit per share
            })

# Sell portfolio: Sell short at max price, cover at min price
for _, row in sell_weights.iterrows():
    if row['portfolio_weight'] > 0:
        asset = row['asset']
        max_price = results_df[results_df['asset'] == asset]['max_price'].iloc[0]
        min_price = results_df[results_df['asset'] == asset]['min_price'].iloc[0]
        allocation = sell_allocation * row['portfolio_weight']
        shares = int(allocation / max_price)  # Integer shares
        if shares > 0:
            cost = shares * max_price
            balance -= cost
            portfolio.append({
                'asset': asset,
                'type': 'sell',
                'shares': shares,
                'buy_price': min_price,  # Cover price
                'sell_price': max_price,  # Short price
                'buy_time': results_df[results_df['asset'] == asset]['min_time'].iloc[0],
                'sell_time': results_df[results_df['asset'] == asset]['max_time'].iloc[0],
                'cost': cost,
                'revenue': shares * (max_price - min_price)  # Profit from short
            })

portfolio_df = pd.DataFrame(portfolio)

In [4]:
# Step 8: Calculate portfolio metrics
risk_free_rate = 0.0408  # Annual risk-free rate

# Function to calculate portfolio metrics
def calculate_portfolio_metrics(portfolio_df, portfolio_type, df_prices):
    if portfolio_type == 'buy':
        df = portfolio_df[portfolio_df['type'] == 'buy']
    else:
        df = portfolio_df[portfolio_df['type'] == 'sell']

    if df.empty:
        return {'return': 0, 'volatility': 0, 'sharpe_ratio': 0}

    total_cost = df['cost'].sum()
    total_revenue = df['revenue'].sum()
    daily_return = total_revenue / total_cost if total_cost > 0 else 0
    # Annualize return for Sharpe ratio
    annualized_return = (1 + daily_return) ** 252 - 1

    # Calculate daily portfolio returns for volatility
    daily_returns = []
    for _, row in df.iterrows():
        asset = row['asset']
        prices = df_prices[asset]
        # Simulate daily return using buy/sell prices
        if row['type'] == 'buy':
            ret = (row['sell_price'] - row['buy_price']) / row['buy_price']
        else:
            ret = (row['sell_price'] - row['buy_price']) / row['sell_price']
        weight = row['cost'] / total_cost if total_cost > 0 else 0
        daily_returns.append(ret * weight)

    if daily_returns:
        daily_portfolio_return = sum(daily_returns)
        # Volatility: std of daily returns, annualized
        volatility = np.std(daily_returns) * np.sqrt(252) if daily_returns else 0
        sharpe_ratio = (annualized_return - risk_free_rate) / volatility if volatility > 0 else 0
    else:
        volatility = 0
        sharpe_ratio = 0

    return {
        'return': daily_return,
        'volatility': volatility,
        'sharpe_ratio': sharpe_ratio
    }

buy_metrics = calculate_portfolio_metrics(portfolio_df, 'buy', df)
sell_metrics = calculate_portfolio_metrics(portfolio_df, 'sell', df)

# Combined portfolio metrics
total_cost = portfolio_df['cost'].sum()
total_revenue = portfolio_df['revenue'].sum()
investment_made = total_investment - balance
portfolio_final_balance = investment_made + total_revenue
overall_return = total_revenue / investment_made if investment_made > 0 else 0
annualized_overall_return = (1 + overall_return) ** 252 - 1
overall_volatility = np.sqrt(buy_metrics['volatility']**2 + sell_metrics['volatility']**2)
overall_sharpe = (annualized_overall_return - risk_free_rate) / overall_volatility if overall_volatility > 0 else 0

# Daily and yearly ROI
daily_roi = overall_return
yearly_roi = (1 + daily_roi) ** 252 - 1  # Compound over 252 trading days

# Step 9: Output results
print("Buy Portfolio:")
print(buy_weights[['asset', 'portfolio_weight']])
print(f"Buy Allocation: ${buy_allocation:.2f}")
print(f"Buy Return: {buy_metrics['return']*100:.4f}%")
print(f"Buy Sharpe Ratio: {buy_metrics['sharpe_ratio']:.4f}")
print("\nSell Portfolio:")
print(sell_weights[['asset', 'portfolio_weight']])
print(f"Sell Allocation: ${sell_allocation:.2f}")
print(f"Sell Return: {sell_metrics['return']*100:.4f}%")
print(f"Sell Sharpe Ratio: {sell_metrics['sharpe_ratio']:.4f}")
print("\nPortfolio Trades:")
print(portfolio_df[['asset', 'type', 'shares', 'buy_price', 'sell_price', 'buy_time', 'sell_time', 'cost', 'revenue']])
print(f"\nInvestment Made: ${investment_made:.2f}")
print(f"Remaining Balance: ${balance:.2f}")
print(f"Portfolio Final Balance: ${portfolio_final_balance:.2f}")
print(f"Overall Return: {overall_return*100:.4f}%")
print(f"Daily ROI: {daily_roi*100:.4f}%")
print(f"Yearly ROI: {yearly_roi*100:.4f}%")
print(f"Overall Portfolio Risk (Volatility): {overall_volatility:.4f}")
print(f"Yearly Sharpe Ratio: {overall_sharpe:.4f}")

# Save portfolio details to a file
portfolio_df.to_csv('portfolio_output.csv', index=False)

Buy Portfolio:
  asset  portfolio_weight
1  SBUX          0.007706
2   JPM          0.032555
3    BA          0.065242
4  ABNB          0.894498
Buy Allocation: $633.22
Buy Return: 0.1169%
Buy Sharpe Ratio: 0.0000

Sell Portfolio:
  asset  portfolio_weight
0  UBER               1.0
Sell Allocation: $366.78
Sell Return: 0.2857%
Sell Sharpe Ratio: 0.0000

Portfolio Trades:
  asset  type  shares   buy_price  sell_price  buy_time  sell_time  \
0  ABNB   buy       4  137.115662  137.275911        20        150   
1  UBER  sell       4   80.861221   81.092890       150          1   

         cost   revenue  
0  548.462648  0.640995  
1  324.371559  0.926673  

Investment Made: $872.83
Remaining Balance: $127.17
Portfolio Final Balance: $874.40
Overall Return: 0.1796%
Daily ROI: 0.1796%
Yearly ROI: 57.1770%
Overall Portfolio Risk (Volatility): 0.0000
Yearly Sharpe Ratio: 0.0000


In [5]:
# prompt: provide the result as csv file as output in computer

files.download('portfolio_output.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>